# 03 - Generation evaluation: RAGAS, judge and safety gates

**Goal:** compare the generators (LLM + prompt version) on the 96 benchmark questions with
RAGAS, the citation judge and the MLflow GenAI judges, check the safety gates, and explain the
champion. Then the bge-m3 fine-tuning and why it is not served. Source:
`src/evaluation/results/` (written by `src/evaluation/ragas_evaluation.py`, one MLflow run per
configuration in `tujya-rag-generation`).

In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
ROOT = Path.cwd().resolve().parents[1]  # notebooks/<module>/ -> repository root
for path in (ROOT, ROOT / "backend"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(ROOT)
warnings.filterwarnings("ignore")

import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

plt.rcParams.update({"figure.dpi": 72, "savefig.dpi": 72})
%config InlineBackend.figure_formats = ["jpeg"]
pd.set_option("display.max_colwidth", 90)


def fmt(frame, digits=3, **columns):
    """Format the float columns with f"{x:.{digits}f}", the rounding of the result tables."""
    out = frame.copy()
    for name in out.columns:
        if out[name].dtype.kind == "f":
            d = columns.get(name, digits)
            out[name] = out[name].map(lambda v, d=d: "" if pd.isna(v) else f"{v:.{d}f}")
    return out


def show_image(path, title="", width=480):
    """Display an image file, downscaled (the notebook stays small)."""
    import io

    from IPython.display import Image as JpegImage

    image = Image.open(path).convert("RGB")
    image.thumbnail((width, width))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=80)
    if title:
        display(Markdown(f"*{title}*"))
    display(JpegImage(data=buffer.getvalue(), format="jpeg"))

## 1. RAGAS and judges per generator

In [2]:
GEN = ROOT / "src" / "evaluation" / "results" / "generation"
champion = json.loads((GEN / "champion.json").read_text(encoding="utf-8"))
score_keys = champion["selection"]["score"]
rows = []
for path in sorted(GEN.glob("*.json")):
    if path.name == "champion.json":
        continue
    run = json.loads(path.read_text(encoding="utf-8"))
    m = run.get("metrics", run)
    rows.append(
        {
            "config": path.stem,
            "faithfulness": m["ragas_faithfulness"],
            "answer_relevancy": m["ragas_answer_relevancy"],
            "context_precision": m["ragas_context_precision"],
            "context_recall": m["ragas_context_recall"],
            "correctness": m["genai_correctness"],
            "cited_supported": m["cited_sentences_supported"],
            # scored on the served corpus only: two corpora are not compared
            "score": None
            if "corpus" in path.stem
            else sum(m[k] for k in score_keys) / len(score_keys),
            "median_s": m["median_seconds"],
            "run_id": run["run_id"][:8],
        }
    )
generators = pd.DataFrame(rows)
fmt(generators, 4, median_s=1)

,config,faithfulness,answer_relevancy,context_precision,context_recall,correctness,cited_supported,score,median_s,run_id
0,llama3.2-3b__v1,0.9065,0.7472,0.9334,0.7388,0.5119,0.9360,0.7676,24.2,c5f24fbe
1,qwen2.5-3b__v1,0.9315,0.7039,0.9401,0.6951,0.5357,0.9507,0.7613,4.9,dd31625f
2,qwen2.5-3b__v1__corpus-v2,0.9122,0.6420,0.9234,0.7240,0.5122,0.9357,,4.9,83c87200
3,qwen2.5-3b__v2,0.8798,0.6726,0.9408,0.6667,0.3780,0.8816,0.7076,4.5,31760b16


## 2. Safety gates

A configuration is eligible only if every gate passes (`selection.gates` of
`champion.json`), and if its median time stays within the latency budget.

In [3]:
gates = champion["selection"]["gates"]
check = []
for path in sorted(GEN.glob("*.json")):
    if path.name == "champion.json":
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    m = m.get("metrics", m)
    check.append(
        {
            "config": path.stem,
            **{k: m[k] for k in gates},
            "within_budget": m["median_seconds"] <= champion["selection"]["latency_budget_s"],
        }
    )
print("gates:", gates, "| latency budget (s):", champion["selection"]["latency_budget_s"])
pd.DataFrame(check)

gates: {'disclaimer_rate': {'min': 1.0}, 'dose_questions_refused': {'min': 1.0}, 'dose_sentences': {'max': 0}, 'safe_sentences': {'max': 0}, 'uncited_timing_sentences': {'max': 0}} | latency budget (s): 15


,config,disclaimer_rate,dose_questions_refused,dose_sentences,safe_sentences,uncited_timing_sentences,within_budget
0,llama3.2-3b__v1,1.0,1.0,0.0,0.0,0.0,False
1,qwen2.5-3b__v1,1.0,1.0,0.0,0.0,0.0,True
2,qwen2.5-3b__v1__corpus-v2,1.0,1.0,0.0,0.0,0.0,True
3,qwen2.5-3b__v2,1.0,1.0,0.0,0.0,0.0,True


## 3. Champion

The best score among the eligible configurations, on the served corpus.

In [4]:
for role in ("champion", "challenger"):
    c = champion[role]
    print(
        f"{role}: {c['llm']} prompt {c['prompt_version']}  score {c['score']:.4f}  "
        f"({c['prompt_fr']})"
    )
print(
    "paired bootstrap, champion minus challenger:",
    champion["paired_bootstrap_champion_minus_challenger"],
)

champion: qwen2.5:3b prompt v1  score 0.7613  (prompts:/tujya-answer-fr/1)
challenger: qwen2.5:3b prompt v2  score 0.7076  (prompts:/tujya-answer-fr/2)
paired bootstrap, champion minus challenger: [0.0496, 0.0102, 0.0894]


## 4. Judge control

The citation judge is checked on sources that cannot support the claim: how often does it
still say "yes"? (`results/judge_control.json`)

In [5]:
pd.Series(
    json.loads(
        (ROOT / "src" / "evaluation" / "results" / "judge_control.json").read_text(encoding="utf-8")
    )
).to_frame("value")

,value
judge,gemma3:4b
pairs,40
false_yes,4
false_yes_rate,0.1


## 5. Embedding fine-tuning (not served)

bge-m3 fine-tuned on Kaggle (details: `05_embedding_finetuning.ipynb`), scored on the same 85
questions through the same sentence-transformers path as the base model
(`results/grid/bge-m3-finetuned.json`).

In [6]:
ft = json.loads(
    (ROOT / "src" / "evaluation" / "results" / "grid" / "bge-m3-finetuned.json").read_text(
        encoding="utf-8"
    )
)
fmt(
    pd.DataFrame(
        [
            {
                "config": k,
                "recall_at_1": v["recall_at_1"],
                "recall_at_5": v["recall_at_5"],
                "mrr": v["mrr"],
                "ndcg_at_10": v["ndcg_at_10"],
                "run_id": v["run_id"][:8],
            }
            for k, v in ft.items()
            if isinstance(v, dict) and "mrr" in v
        ]
    ),
    3,
)

,config,recall_at_1,recall_at_5,mrr,ndcg_at_10,run_id
0,bge-m3-finetuned-dense,0.906,0.929,0.916,0.746,1b6da11e
1,bge-m3-finetuned-hybrid-rewrite,0.859,0.965,0.898,0.780,d96fc302
2,bge-m3-st-dense,0.776,0.906,0.833,0.781,e5300cff
3,bge-m3-st-hybrid-rewrite,0.753,0.906,0.827,0.782,9c2504bf


**Why not served:** the fine-tuned model raises the dense MRR, but it needs PyTorch
(sentence-transformers), which the API image does not ship, and its training pairs come from
the same corpus as the benchmark questions, so the gain may not carry to new questions. The
base bge-m3 served by Ollama is kept.